# TeReDiff · GIRAFFE

Проверен на Kaggle T4 на авторском демо и четырёх примерах GIRAFFE при 512×512: результат визуально близок к опубликованному, но не проверялся на побитовое совпадение.

Запускайте сверху вниз в отдельной сессии Kaggle с GPU и Internet. Зависимости устанавливаются в отдельное Python-окружение.

Подключите датасет через Add Input и укажите его путь в разделе 4. Структура: `<корень>/<sample_id>/degraded.png`; `hint.png` нужен только для просмотра.

[Официальный репозиторий](https://github.com/cvlab-kaist/TAIR) · [Зафиксированная версия](https://github.com/cvlab-kaist/TAIR/tree/5f0365f4bf37d7ea1bad2544e988b28f838b0c08)

## 1. Репозиторий


In [ ]:
from pathlib import Path
import datetime
import hashlib
import json
import os
import shutil
import struct
import subprocess
import sys
import uuid
import zipfile

REPO_URL = 'https://github.com/cvlab-kaist/TAIR.git'
COMMIT = '5f0365f4bf37d7ea1bad2544e988b28f838b0c08'
WORK_ROOT = Path('/kaggle/working/giraffe_baselines') if Path('/kaggle/working').exists() else Path.cwd() / '.baseline_work'
WORK_ROOT = WORK_ROOT.resolve()
REPO_DIR = WORK_ROOT / 'external' / 'terediff'

def run(*args, cwd=REPO_DIR, env=None):
    subprocess.run([str(a) for a in args], cwd=cwd, env=env, check=True)

REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
if not REPO_DIR.exists():
    run('git', 'clone', REPO_URL, REPO_DIR, cwd=WORK_ROOT)
origin = subprocess.check_output(['git', '-C', str(REPO_DIR), 'remote', 'get-url', 'origin'], text=True).strip()
if origin.removesuffix('.git') != REPO_URL.removesuffix('.git'):
    raise RuntimeError(f'В {REPO_DIR} находится другой репозиторий: {origin}')
if subprocess.check_output(['git', '-C', str(REPO_DIR), 'diff', 'HEAD', '--name-only'], text=True).strip():
    raise RuntimeError('В baseline-репозитории изменены отслеживаемые файлы. Сохраните изменения отдельно.')
run('git', 'fetch', 'origin', COMMIT)
run('git', 'checkout', '--detach', COMMIT)
actual = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
if actual != COMMIT:
    raise RuntimeError(f'Версия репозитория не совпала: {actual}')
print(f'terediff: {actual}')


## 2. Окружение

В ячейке инференса сохранён путь восстановления из официального `val.py`: SwinIR, диффузия, 50 шагов, CFG=1 и промпт из OCR. Загрузка GT и метрики исключены. Модель получает входы 512×512. Авторский режим проверен на GPU. Режим патчей ещё не проверен на GPU. Stage 3 checkpoint содержит TESTR. Установка Detectron2/TESTR требует CUDA Toolkit и собирает CUDA-расширения.


In [ ]:
# Зависимости метода устанавливаются отдельно от ядра Jupyter.
if sys.platform != 'linux' or not shutil.which('nvidia-smi'):
    raise RuntimeError('Нужна Linux-сессия с NVIDIA GPU. В Kaggle включите GPU и Internet. На Mac CUDA недоступна.')
run('nvidia-smi', cwd=WORK_ROOT)
run(sys.executable, '-m', 'pip', 'install', 'uv==0.9.7', cwd=WORK_ROOT)
ENV_DIR = WORK_ROOT / 'envs' / 'terediff'
if not (ENV_DIR / 'bin/python').exists():
    run(sys.executable, '-m', 'uv', 'venv', '--python', '3.10', '--seed', ENV_DIR, cwd=WORK_ROOT)
PYTHON = ENV_DIR / 'bin/python'

def run_python(source, *args, env=None):
    run(PYTHON, '-c', source, *args, env=env)

run(PYTHON, '-m', 'pip', 'install', 'pip==24.3.1', 'setuptools==75.6.0', 'wheel==0.45.1')
run(PYTHON, '-m', 'pip', 'install', 'torch==2.2.2', 'torchvision==0.17.2', 'torchaudio==2.2.2', '--index-url', 'https://download.pytorch.org/whl/cu121')
run(PYTHON, '-m', 'pip', 'install', '-r', 'requirements.txt', 'numpy==1.26.4',
    'huggingface-hub==0.25.2', 'gdown==5.2.0', 'ninja==1.11.1.3')
# Системная CUDA Kaggle может отличаться от CUDA 12.1 в PyTorch.
import tarfile
import urllib.request

TOOLCHAIN = WORK_ROOT / 'envs' / 'terediff_cuda121'
MAMBA = WORK_ROOT / 'tools' / 'bin' / 'micromamba'
if not MAMBA.is_file():
    archive = WORK_ROOT / 'micromamba.tar.bz2'
    urllib.request.urlretrieve('https://micro.mamba.pm/api/micromamba/linux-64/latest', archive)
    with tarfile.open(archive) as bundle:
        bundle.extract('bin/micromamba', path=MAMBA.parent.parent, filter='data')
    archive.unlink()
mamba_env = dict(os.environ, MAMBA_ROOT_PREFIX=str(WORK_ROOT / 'mamba'))
run(MAMBA, 'install' if (TOOLCHAIN / 'conda-meta').is_dir() else 'create',
    '-y', '-p', TOOLCHAIN, '--override-channels', '-c', 'nvidia', '-c', 'conda-forge',
    'cuda-nvcc=12.1.105', 'cuda-cudart=12.1.105',
    'cuda-cudart-dev=12.1.105', 'cuda-cccl=12.1.109',
    'gxx_linux-64=11', env=mamba_env)
# Заголовки cuSPARSE/cuBLAS/cuSOLVER уже поставлены CUDA-пакетами PyTorch.
nvidia_includes = sorted((ENV_DIR / 'lib/python3.10/site-packages/nvidia').glob('*/include'))
if not any((path / 'cusparse.h').is_file() for path in nvidia_includes):
    raise RuntimeError('Не найден cusparse.h в CUDA-пакетах Python-окружения.')
build_env = dict(os.environ,
    CPATH=os.pathsep.join([str(path) for path in nvidia_includes] +
                         ([os.environ['CPATH']] if os.environ.get('CPATH') else [])),
    CUDA_HOME=str(TOOLCHAIN),
    PATH=str(TOOLCHAIN / 'bin') + os.pathsep + os.environ['PATH'],
    CC=str(TOOLCHAIN / 'bin/x86_64-conda-linux-gnu-cc'),
    CXX=str(TOOLCHAIN / 'bin/x86_64-conda-linux-gnu-c++'),
    MAX_JOBS='2')
run(TOOLCHAIN / 'bin/nvcc', '--version', env=build_env)
run(PYTHON, '-m', 'pip', 'install', '-v', '--no-build-isolation', '-e', str(REPO_DIR / 'detectron2'), env=build_env)
run(PYTHON, '-m', 'pip', 'install', '-v', '--no-build-isolation', '-e', str(REPO_DIR / 'testr'), env=build_env)
run_python("import torch; assert torch.cuda.is_available(), 'PyTorch не видит CUDA'; print(torch.__version__, torch.cuda.get_device_name(0))")
run(PYTHON, '-m', 'pip', 'check')


## 3. Веса

Веса DiffBIR скачиваются с Hugging Face, `terediff_stage3.pt` — с Google Drive. При ограничении Google Drive загрузите файл вручную по пути из сообщения об ошибке.

Загрузка начинается при запуске ячейки. Уже скачанные файлы используются повторно.


In [ ]:
run_python(r"""from pathlib import Path
import json, shutil, hashlib, filecmp
from huggingface_hub import hf_hub_download
import gdown
assets = []

def hf_file(repo, filename, destination, revision):
    destination = Path(destination)
    src = Path(hf_hub_download(repo_id=repo, filename=filename, revision=revision)).resolve()
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.is_symlink():
        if destination.exists() and not filecmp.cmp(src, destination, shallow=False):
            raise RuntimeError(f'Файл отличается от закреплённых весов: {destination}')
        # Одна копия весов в HF-кэше вместо дублирования нескольких гигабайт.
        link = destination.with_name(destination.name + '.link')
        link.symlink_to(src)
        link.replace(destination)
    assets.append({'source': f'https://huggingface.co/{repo}', 'revision': revision,
                   'file': filename, 'local': str(destination), 'bytes': destination.stat().st_size})
    return destination


def drive_file(folder_url, filename, destination):
    destination = Path(destination)
    if not destination.is_file() or destination.stat().st_size == 0:
        entries = gdown.download_folder(url=folder_url, skip_download=True, quiet=True)
        matches = [entry for entry in (entries or []) if Path(entry.path).name == filename]
        if len(matches) != 1:
            raise RuntimeError(f'Не найден однозначный {filename}. Скачайте его вручную из {folder_url} и положите в {destination}; затем повторите ячейку.')
        destination.parent.mkdir(parents=True, exist_ok=True)
        part = destination.with_name(destination.name + '.part')
        if not gdown.download(id=matches[0].id, output=str(part), quiet=False, resume=True):
            raise RuntimeError(f'Google Drive не отдал {filename}. Загрузите файл вручную в {destination}.')
        part.replace(destination)
    with destination.open('rb') as f:
        digest = hashlib.file_digest(f, 'sha256').hexdigest() if hasattr(hashlib, 'file_digest') else None
    if digest is None:
        h = hashlib.sha256()
        with destination.open('rb') as f:
            for chunk in iter(lambda: f.read(8 * 1024 * 1024), b''):
                h.update(chunk)
        digest = h.hexdigest()
    assets.append({'source': folder_url, 'file': filename, 'local': str(destination),
                   'bytes': destination.stat().st_size, 'sha256': digest})
    return destination

for name in ['realesrgan_s4_swinir_100k.pth', 'DiffBIR_v2.1.pt', 'sd2.1-base-zsnr-laionaes5.ckpt']:
    hf_file('lxq007/DiffBIR-v2', name, 'weights/' + name, 'fd6263503564bfbfa156665a90832e1b729434f4')
drive_file('https://drive.google.com/drive/folders/1Xn0DaL-3ViXpl1pWHPvcmSejTDoIjAQn',
           'terediff_stage3.pt', 'weights/terediff_stage3.pt')
# Stage 3 содержит совместно обученные cldm/swinir/testr: отдельные training-веса TESTR не нужны.
import torch
state = torch.load('weights/terediff_stage3.pt', map_location='cpu')
missing = {'cldm', 'swinir', 'testr'} - set(state)
if missing:
    raise RuntimeError(f'Неполный stage3 checkpoint: отсутствуют {sorted(missing)}')
del state
Path('giraffe_assets.json').write_text(json.dumps({'assets': assets}, indent=2))
""")


## 4. Параметры и данные

Вход модели — только `degraded.png`. Каждое выполнение создаёт новую папку запуска.

`tiled` обрабатывает патчи 512×512. Перекрытие минимум 128 пикселей и может быть больше у краёв. Если длинная сторона меньше 512, изображение пропорционально увеличивается. Недостающие пиксели по краям дополняются, после сборки отступы удаляются. Итог сохраняется в рабочем разрешении после увеличения. Режим `author` приводит весь кадр к 512×512.


In [ ]:
DATASET_ROOT = Path('/kaggle/input/datasets/markunyaa/giraffe-dataset/giraffe_dataset')
RESULTS_ROOT = WORK_ROOT / 'results'
SEED = 25
MODE = 'tiled'  # 'author' — ресайз всего кадра в 512×512.
if MODE not in ('tiled', 'author'):
    raise ValueError(f'Неизвестный режим: {MODE}')
RESULT_NAME = 'terediff_tiled' if MODE == 'tiled' else 'terediff'
SETTINGS = {
    'mode': MODE,
    'input_resize': [512, 512] if MODE == 'author' else None,
    'tile_size': 512,
    'tile_overlap': 128,
    'pad_mode': 'replicate',
    'steps': 50,
    'cfg_scale': 1.0,
    'text_score_threshold': 0.5,
    'prompt_style': 'CAPTION',
    'initial_prompt': '',
}

DATASET_ROOT = DATASET_ROOT.expanduser().resolve()
RESULTS_ROOT = RESULTS_ROOT.expanduser().resolve()
sources = sorted(DATASET_ROOT.glob('*/degraded.png'))
if not sources:
    raise FileNotFoundError(f'Не найдены */degraded.png в {DATASET_ROOT}. Укажите путь из Kaggle Add Input.')
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
RUN_DIR = WORK_ROOT / 'runs' / RESULT_NAME / RUN_ID
INPUT_DIR = RUN_DIR / 'input'
RAW_DIR = RUN_DIR / 'raw'
INPUT_DIR.mkdir(parents=True)
RAW_DIR.mkdir()
SAMPLES = {}
for src in sources:
    sample_id = src.parent.name
    target = INPUT_DIR / f'{sample_id}.png'
    shutil.copy2(src, target)
    SAMPLES[sample_id] = src
ASSETS = json.loads((REPO_DIR / 'giraffe_assets.json').read_text())
RUN_CONFIG = {'method': 'terediff', 'repo': REPO_URL, 'commit': COMMIT, 'seed': SEED,
              'settings': SETTINGS, 'weights': ASSETS, 'dataset_root': str(DATASET_ROOT),
              'input_dir': str(INPUT_DIR), 'raw_dir': str(RAW_DIR), 'run_id': RUN_ID}
CONFIG_PATH = RUN_DIR / 'run.json'
CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, indent=2, ensure_ascii=False))
INFERENCE_OK = False
EXPORT_OK = False
print(f'{len(SAMPLES)} примеров: ' + ', '.join(SAMPLES))
print(f'Результаты: {RESULTS_ROOT}')


## 5. Инференс

Для повторного запуска сначала выполните ячейку параметров.


In [ ]:
INFERENCE_OK = False
EXPORT_OK = False
if any(RAW_DIR.iterdir()):
    raise RuntimeError('Для повторного инференса заново выполните ячейку параметров: она создаст новую папку запуска.')
TEREDIFF_INFERENCE = r"""from pathlib import Path
from types import SimpleNamespace
import json, sys
import torch
from torch import nn
from PIL import Image
from torchvision import transforms as T
from torchvision.transforms.functional import to_pil_image
from accelerate import Accelerator
from accelerate.utils import set_seed, DistributedDataParallelKwargs
from omegaconf import OmegaConf
import initialize
from terediff.utils.common import instantiate_from_config
from terediff.sampler import SpacedSampler

job = json.loads(Path(sys.argv[1]).read_text())
s = job['settings']
accelerator = Accelerator(split_batches=False, kwargs_handlers=[DistributedDataParallelKwargs(find_unused_parameters=True)])
device = accelerator.device
set_seed(job['seed'], device_specific=False)
generator = torch.Generator(device).manual_seed(job['seed'])
cfg = OmegaConf.load('configs/val/val_terediff.yaml')
cfg.exp_args.resume_ckpt_dir = str(Path('weights/terediff_stage3.pt').resolve())
cfg.exp_args.testr_ckpt_dir = None
cfg.exp_args.prompt_style = s['prompt_style']
cfg.log_args.log_tool = None
args = SimpleNamespace(config_testr='testr/configs/TESTR/TESTR_R_50_Polygon.yaml')
models, _ = initialize.load_model(accelerator, device, args, cfg)
diffusion = instantiate_from_config(cfg.model.diffusion).to(device)
sampler = SpacedSampler(diffusion.betas, diffusion.parameterization, rescale_cfg=False)
models = {k: accelerator.prepare(v) for k,v in models.items()}
pure_cldm = accelerator.unwrap_model(models['cldm'])
for model in models.values():
    if isinstance(model, nn.Module):
        model.eval()
size = s['tile_size']
overlap = s['tile_overlap']
to_tensor = T.ToTensor()

@torch.no_grad()
def restore_tile(tile):
    lq = tile.unsqueeze(0).to(device)
    val_prompt = [s['initial_prompt']]
    clean = models['swinir'](lq)
    cond = pure_cldm.prepare_condition(clean, val_prompt)
    latent_shape = (1, 4, size // 8, size // 8)
    noise = torch.randn(latent_shape, generator=generator, device=device, dtype=torch.float32)
    models['testr'].test_score_threshold = s['text_score_threshold']
    z, text_results = sampler.val_sample(
        model=models['cldm'], device=device, steps=s['steps'], x_size=latent_shape,
        cond=cond, uncond=None, cfg_scale=s['cfg_scale'], x_T=noise,
        progress=True, cfg=cfg, pure_cldm=pure_cldm, ts_model=models['testr'],
        val_prompt=val_prompt)
    restored = ((pure_cldm.vae_decode(z) + 1) / 2).clamp(0, 1)
    transcript = [
        {'timestep': int(item['timestep']), 'pred_texts': list(item['pred_texts'])}
        for item in text_results
    ]
    return restored[0].float().cpu(), transcript

def tile_starts(length):
    span = length - size
    stride = size - overlap
    intervals = (span + stride - 1) // stride
    if not intervals:
        return [0]
    return [span * i // intervals for i in range(intervals + 1)]

def axis_weight(starts, i):
    weight = torch.ones(size)
    if i > 0:
        shared = size - (starts[i] - starts[i - 1])
        weight[:shared] *= torch.arange(1, shared + 1) / (shared + 1)
    if i + 1 < len(starts):
        shared = size - (starts[i + 1] - starts[i])
        weight[-shared:] *= torch.arange(shared, 0, -1) / (shared + 1)
    return weight

for path in sorted(Path(job['input_dir']).glob('*.png')):
    with Image.open(path) as image:
        rgb = image.convert('RGB')
    if s['mode'] == 'author':
        height, width = s['input_resize']
        restored, transcript = restore_tile(
            to_tensor(rgb.resize((width, height), Image.Resampling.BICUBIC)))
    else:
        scale = max(1, size / max(rgb.size))
        width, height = (max(1, round(dim * scale)) for dim in rgb.size)
        if (width, height) != rgb.size:
            rgb = rgb.resize((width, height), Image.Resampling.BICUBIC)
        padded = torch.nn.functional.pad(
            to_tensor(rgb).unsqueeze(0),
            (0, max(0, size - width), 0, max(0, size - height)),
            mode=s['pad_mode'])[0]
        ys = tile_starts(padded.shape[-2])
        xs = tile_starts(padded.shape[-1])
        merged = torch.zeros_like(padded)
        weights = torch.zeros_like(padded[:1])
        transcript = []
        print(f'{path.name}: {width}x{height}, {len(ys) * len(xs)} patches', flush=True)
        for row, y in enumerate(ys):
            wy = axis_weight(ys, row)
            for col, x in enumerate(xs):
                region = (slice(None), slice(y, y + size), slice(x, x + size))
                restored_tile, tile_ocr = restore_tile(padded[region])
                weight = wy[:, None] * axis_weight(xs, col)[None, :]
                merged[region] += restored_tile * weight
                weights[region] += weight
                transcript.append({'x': x, 'y': y, 'steps': tile_ocr})
        restored = (merged / weights)[:, :height, :width]

    to_pil_image(restored).save(Path(job['raw_dir']) / path.name)
    (Path(job['raw_dir']) / (path.stem + '_ocr.json')).write_text(
        json.dumps(transcript, ensure_ascii=False, indent=2))
"""
COMMAND = [str(PYTHON), '-c', TEREDIFF_INFERENCE, str(CONFIG_PATH)]
run(*COMMAND)
PREDICTIONS = {sid: RAW_DIR / f'{sid}.png' for sid in SAMPLES}
INFERENCE_OK = True


## 6. Сохранение

Результаты: `results/<sample_id>/terediff_tiled.png` для патчей и `results/<sample_id>/terediff.png` для авторского режима. Рядом сохраняются параметры, версии весов, seed и размеры изображений. Архив содержит результаты и сведения о запуске. Повторный экспорт заменяет PNG и JSON этого метода; архивы сохраняются отдельно.


In [ ]:
EXPORT_OK = False
if not INFERENCE_OK:
    raise RuntimeError('Сначала выполните инференс текущего запуска без ошибок.')
missing = [str(p) for p in PREDICTIONS.values() if not p.is_file()]
if missing:
    raise RuntimeError('Метод не создал все результаты: ' + ', '.join(missing))

def png_size(path):
    with Path(path).open('rb') as f:
        header = f.read(24)
    if header[:8] != b'\x89PNG\r\n\x1a\n':
        raise ValueError(f'Ожидался PNG: {path}')
    return list(struct.unpack('>II', header[16:24]))

for path in PREDICTIONS.values():
    png_size(path)
freeze = subprocess.check_output([str(PYTHON), '-m', 'pip', 'freeze'], text=True)
(RUN_DIR / 'environment.txt').write_text(freeze)
RUN_CONFIG['command'] = COMMAND
RUN_CONFIG['status'] = 'completed'
RUN_CONFIG['completed_at_utc'] = datetime.datetime.now(datetime.timezone.utc).isoformat()
CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, indent=2, ensure_ascii=False))
EXPORTED = []
for sid, predicted in PREDICTIONS.items():
    out_dir = RESULTS_ROOT / sid
    out_dir.mkdir(parents=True, exist_ok=True)
    output = out_dir / f'{RESULT_NAME}.png'
    shutil.copy2(predicted, output)
    metadata = dict(RUN_CONFIG, sample_id=sid,
        input_size=png_size(INPUT_DIR / f'{sid}.png'), output_size=png_size(output),
        input_sha256=hashlib.sha256((INPUT_DIR / f'{sid}.png').read_bytes()).hexdigest(),
        output_sha256=hashlib.sha256(output.read_bytes()).hexdigest())
    info = out_dir / f'{RESULT_NAME}.json'
    info.write_text(json.dumps(metadata, indent=2, ensure_ascii=False))
    EXPORTED.extend([output, info])
archive = WORK_ROOT / f'giraffe_{RESULT_NAME}_{RUN_ID}.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for path in EXPORTED:
        z.write(path, Path('results') / path.relative_to(RESULTS_ROOT))
    z.write(CONFIG_PATH, 'run.json')
    z.write(RUN_DIR / 'environment.txt', 'environment.txt')
    for path in RAW_DIR.glob('*_ocr.json'):
        z.write(path, Path('ocr') / path.name)
EXPORT_OK = True
print(f'Сохранено {len(PREDICTIONS)} результатов: {archive}')


## 7. Просмотр

`hint.png` — референс для сравнения. Он не поступает в модель и может отличаться ракурсом или кадрированием.


In [ ]:
if not EXPORT_OK:
    raise RuntimeError('Сначала завершите сохранение результатов текущего запуска.')
from IPython.display import display, HTML, FileLink
import base64
import html
for sid, source in SAMPLES.items():
    items = [('Вход: degraded.png', source), (f'Результат: {RESULT_NAME}', RESULTS_ROOT / sid / f'{RESULT_NAME}.png')]
    hint = source.parent / 'hint.png'
    if hint.is_file():
        items.append(('Референс (hint.png)', hint))
    panels = []
    for label, path in items:
        encoded = base64.b64encode(path.read_bytes()).decode('ascii')
        panels.append(f'<div style="max-width:32%;"><p>{html.escape(label)}</p><img style="width:100%;height:auto" src="data:image/png;base64,{encoded}"></div>')
    display(HTML('<h3>' + html.escape(sid) + '</h3><div style="display:flex;gap:12px;align-items:flex-start">' + ''.join(panels) + '</div>'))
# Относительная ссылка обслуживается Jupyter; в Kaggle архив также виден в Output.
display(FileLink(os.path.relpath(archive, Path.cwd())))
